In [ ]:
import numpy as np
import os

In [ ]:
# sims_number = 100
# Probably won't change

username                        = "jsolisle"
platform = "archer2"

# Particular experiment

folder_experiment_name_archer = "HCM/4/scenarios/49"
num_beats_limit_cycle  = 500
num_beats              = 5
# bpm                    = 72
# bpm                    = 60
# bcl                    = int(60000/bpm) ### Change it also in the clinical_data json file

# HPC
# /work/e348/e348/<archer2_username>/rodero_healthy/h11/scenarios/5/states
# states_folder = f"/rds/general/user/{username}/home/{folder_experiment_name}/states" # put here the .sv files on the hpc
harddrive = "/scratch-nvme/e348/e348"
states_folder = f"/{harddrive}/{username}/{folder_experiment_name_archer}/states" # put here the .sv files on the hpc
# states_folder = f"/work/e348/e348/{username}/everyones_states" # put here the .sv files on the hpc
tags_setup_file_path_archer2    = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/tags.json"

# Local computer
# base_folder_local   = f"/path/to/data/SeagateExpansionDrive/{folder_experiment_name}"
local_hard_drive="/path/to/data/Elements"
folder_experiment_name = "HCM/4/scenarios/49_more_samples"
base_folder_local   = f"/{local_hard_drive}/{folder_experiment_name}"
cell_sims_basefolder = f"{base_folder_local}/cell_sims"
torord_folder       = f"{base_folder_local}/SS/param"
courtemanche_folder = f"{base_folder_local}/SS/param"
data_folder          = f"{base_folder_local}/data"
json_folder = f"{base_folder_local}/json_files"

general_setup_file_path_local = f"{json_folder}/{platform}_setup.json"
slrm_folder  = f"{base_folder_local}/slrm"

os.makedirs(cell_sims_basefolder, exist_ok=True)

# Merge the normalised datasets

In [ ]:

fields = ['ToRORd',
          'ToRORd_land',
          'COURTEMANCHE',
          'COURTEMANCHE_land',
          'EP',
          'circadapt',
          'mechanics']

X_array = []
xlabels_array = []

for field in fields:
    if field == "EP" or field == "mechanics":
        X_ = np.loadtxt(f"{data_folder}/X_{field}_normalised.txt", dtype=float)
    else:
        X_ = np.loadtxt(f"{data_folder}/X_{field}.txt", dtype=float)

    # Check if X_ has only one column, reshape to 2D array
    if X_.ndim == 1:
        X_ = X_.reshape(-1, 1)

    X_array.append(X_)

    xlabels_ = []
    with open(f"{data_folder}/xlabels_{field}.txt", "r") as f:
        for line in f:
            xlabels_.append(line.replace("\n", ""))

    xlabels_array.append(xlabels_)

# X = np.concatenate(X_array, axis=1)
    
X = np.hstack(X_array)
xlabels = np.concatenate(xlabels_array, axis=0)

np.savetxt(f"{data_folder}/X_with_normalised_values.txt",X,fmt="%g")
np.savetxt(f"{data_folder}/xlabels.txt",xlabels,fmt="%s")

# We scale the EP data based on the patient-specific intervals

In [ ]:
os.system(f"python3 ../simulation_toolbox/patient_specific_CV.py --basefolder {base_folder_local}")

# We generate the simulation json files needed

In [ ]:


# fields = ['COURTEMANCHE_land'
#           ]



cmd = "python ../simulation_toolbox/generate_json_parameter_files.py"
cmd += f" --datafolder {data_folder}"
cmd += f" --fields {' '.join(fields)}"
cmd += f" --paramfolder {json_folder}"
cmd += f" --defaultfile {json_folder}/default.json"
os.system(cmd)

# Merge all the dataset fields

In [ ]:
X_array = []
xlabels_array = []

for field in fields:
    X_ = np.loadtxt(f"{data_folder}/X_{field}.txt", dtype=float)

    # Check if X_ has only one column, reshape to 2D array
    if X_.ndim == 1:
        X_ = X_.reshape(-1, 1)

    X_array.append(X_)

    xlabels_ = []
    with open(f"{data_folder}/xlabels_{field}.txt", "r") as f:
        for line in f:
            xlabels_.append(line.replace("\n", ""))

    xlabels_array.append(xlabels_)

# X = np.concatenate(X_array, axis=1)
    
X = np.hstack(X_array)
xlabels = np.concatenate(xlabels_array, axis=0)

np.savetxt(f"{data_folder}/X.txt",X,fmt="%g")
np.savetxt(f"{data_folder}/xlabels.txt",xlabels,fmt="%s")

# We generate the cycle slrm files

In [ ]:
first_sim = 800
last_sim = 6199
cmd = "python ../simulation_toolbox/write_simulation_scripts.py"
cmd += f" --datafolder {data_folder}"
cmd += f" --setup_file {general_setup_file_path_local}"
cmd += f" --paramfolder {json_folder}"
cmd += f" --slrmfolder {slrm_folder}"
cmd += f" --HPC_statefolder {states_folder}"
cmd += f" --idx1 {first_sim}"
cmd += f" --idx2 {last_sim}"
cmd += f" --tags_file {base_folder_local}/json_files/tags_lvrv_fch.json"
cmd += f" --clinical_data {base_folder_local}/json_files/clinical_data.json"
cmd += f" --cell_sims_folder {base_folder_local}/SS"

os.system(cmd)